# Enhanced Crop Recommendation System
### Using Machine Learning & Deep Learning
**Algorithms:** Logistic Regression | Naïve Bayes | KNN | Decision Tree | Random Forest | XGBoost | ANN

## Step 1 — Install Dependencies

In [ ]:
!pip install xgboost --quiet
print('All dependencies ready.')

## Step 2 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

# Set consistent plot style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')
print('Libraries imported successfully.')

## Step 3 — Load Dataset

In [ ]:
# Upload the CSV when prompted (Crop_recommendation.csv)
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv('Crop_recommendation.csv')
print(f'Dataset shape: {df.shape}')
print(f'Crop classes: {df["label"].nunique()}')
print(f'\nFirst 5 rows:')
df.head()

## Step 4 — Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Basic Statistics ===')
df.describe().round(2)

In [ ]:
# ── Fig 1: Crop Class Distribution ──────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
crop_counts = df['label'].value_counts()
colors = sns.color_palette('tab20', len(crop_counts))
bars = ax.bar(crop_counts.index, crop_counts.values, color=colors, edgecolor='white', linewidth=0.8)
ax.set_title('Fig. 1 — Crop Class Distribution in Dataset', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Crop Label', fontsize=11)
ax.set_ylabel('Number of Samples', fontsize=11)
ax.tick_params(axis='x', rotation=45)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.savefig('fig1_crop_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 2: Feature Distributions ─────────────────────────────
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
palette = sns.color_palette('Set2', 7)
for i, feat in enumerate(features):
    sns.histplot(df[feat], kde=True, ax=axes[i], color=palette[i], edgecolor='white')
    axes[i].set_title(f'Distribution of {feat}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(feat, fontsize=9)
    axes[i].set_ylabel('Count', fontsize=9)
axes[-1].set_visible(False)
fig.suptitle('Fig. 2 — Feature Distributions (N, P, K, Temperature, Humidity, pH, Rainfall)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig2_feature_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 3: Boxplots for Outlier Detection ─────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
palette = sns.color_palette('Pastel1', 7)
for i, feat in enumerate(features):
    sns.boxplot(y=df[feat], ax=axes[i], color=palette[i])
    axes[i].set_title(f'{feat}', fontsize=11, fontweight='bold')
    axes[i].set_ylabel(feat, fontsize=9)
axes[-1].set_visible(False)
fig.suptitle('Fig. 3 — Boxplots for Outlier Detection Across All Features',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig3_boxplots.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 4: Correlation Heatmap ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
numeric_df = df[features]
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8},
            ax=ax, annot_kws={'size': 9})
ax.set_title('Fig. 4 — Correlation Heatmap of Input Features', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('fig4_correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 5: Pairplot of Key Features ──────────────────────────
# Sample for faster rendering
df_sample = df.sample(300, random_state=42)
g = sns.pairplot(df_sample[['N','P','K','temperature','label']],
                 hue='label', plot_kws={'alpha': 0.5, 's': 20},
                 diag_kind='kde', height=2.0)
g.fig.suptitle('Fig. 5 — Pairplot: N, P, K, Temperature by Crop Class',
               fontsize=12, fontweight='bold', y=1.01)
g.fig.savefig('fig5_pairplot.png', bbox_inches='tight')
plt.show()

## Step 5 — Preprocessing

In [ ]:
# Remove duplicates and encode labels
df.drop_duplicates(inplace=True)
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])
print('Crop classes after encoding:')
for i, cls in enumerate(le.classes_):
    print(f'  {i:2d} -> {cls}')

In [ ]:
X = df[features]
y = df['label_enc']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training samples : {X_train.shape[0]}')
print(f'Testing  samples : {X_test.shape[0]}')
print(f'Features         : {X_train.shape[1]}')

## Step 6 — Train ML Models

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_acc = accuracy_score(y_test, lr_pred)
print(f'Logistic Regression Accuracy: {lr_acc*100:.4f}%')

In [ ]:
# Naïve Bayes
nb = GaussianNB()
nb.fit(X_train, y_train)
nb_pred = nb.predict(X_test)
nb_acc = accuracy_score(y_test, nb_pred)
print(f'Naïve Bayes Accuracy: {nb_acc*100:.4f}%')

In [ ]:
# K-Nearest Neighbor
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)
knn_pred = knn.predict(X_test)
knn_acc = accuracy_score(y_test, knn_pred)
print(f'KNN Accuracy: {knn_acc*100:.4f}%')

In [ ]:
# Decision Tree
dt = DecisionTreeClassifier(max_depth=20, random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
dt_acc = accuracy_score(y_test, dt_pred)
print(f'Decision Tree Accuracy: {dt_acc*100:.4f}%')

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=500, max_depth=20, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
print(f'Random Forest Accuracy: {rf_acc*100:.4f}%')

In [ ]:
# XGBoost
xgb = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=8,
                    use_label_encoder=False, eval_metric='mlogloss', random_state=42)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_pred)
print(f'XGBoost Accuracy: {xgb_acc*100:.4f}%')

## Step 7 — Artificial Neural Network (ANN)

In [ ]:
n_classes = len(le.classes_)

ann = Sequential([
    Dense(256, activation='relu', input_dim=X_train.shape[1]),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(n_classes, activation='softmax')
])

ann.compile(optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'])

ann.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = ann.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

ann_loss, ann_acc = ann.evaluate(X_test, y_test, verbose=0)
print(f'\nANN Test Accuracy: {ann_acc*100:.4f}%')

In [ ]:
# ── Fig 6: ANN Training History ──────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(history.history['accuracy'], label='Train Accuracy', color='#2196F3', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Val Accuracy', color='#FF5722',
         linewidth=2, linestyle='--')
ax1.set_title('Model Accuracy over Epochs', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(history.history['loss'], label='Train Loss', color='#4CAF50', linewidth=2)
ax2.plot(history.history['val_loss'], label='Val Loss', color='#9C27B0',
         linewidth=2, linestyle='--')
ax2.set_title('Model Loss over Epochs', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

fig.suptitle('Fig. 6 — ANN Training & Validation Accuracy/Loss Curves',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig6_ann_training_history.png', bbox_inches='tight')
plt.show()

## Step 8 — Model Comparison & Visualization

In [ ]:
# Summary table
results = pd.DataFrame({
    'Algorithm': ['Logistic Regression', 'Naïve Bayes', 'KNN', 'Decision Tree', 'Random Forest', 'XGBoost', 'ANN'],
    'Accuracy (%)': [
        round(lr_acc*100, 4), round(nb_acc*100, 4), round(knn_acc*100, 4),
        round(dt_acc*100, 4), round(rf_acc*100, 4), round(xgb_acc*100, 4),
        round(ann_acc*100, 4)
    ]
}).sort_values('Accuracy (%)', ascending=False).reset_index(drop=True)

print('=== Model Accuracy Comparison ===')
print(results.to_string(index=False))
print(f'\nBest Model: {results.iloc[0]["Algorithm"]} ({results.iloc[0]["Accuracy (%)"]:.4f}%)')

In [ ]:
# ── Fig 7: Enhanced Model Accuracy Comparison Bar Chart ───────
fig, ax = plt.subplots(figsize=(13, 6))

algo_names = results['Algorithm'].tolist()
acc_values = results['Accuracy (%)'].tolist()
colors = ['#FFD700' if v == max(acc_values) else '#4472C4' for v in acc_values]

bars = ax.bar(algo_names, acc_values, color=colors, edgecolor='white',
              linewidth=1.2, width=0.6)

# Value labels on bars
for bar, val in zip(bars, acc_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylim(min(acc_values) - 1, 101)
ax.set_title('Fig. 7 — Model Accuracy Comparison (All Classifiers)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Algorithm', fontsize=11)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', alpha=0.3)

gold_patch = mpatches.Patch(color='#FFD700', label='Best Model')
blue_patch = mpatches.Patch(color='#4472C4', label='Other Models')
ax.legend(handles=[gold_patch, blue_patch], fontsize=10)

plt.tight_layout()
plt.savefig('fig7_model_comparison_bar.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 8: Horizontal Ranked Bar Chart ────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
sorted_results = results.sort_values('Accuracy (%)')
colors_h = ['#FFD700' if v == max(acc_values) else '#5B9BD5'
            for v in sorted_results['Accuracy (%)']]
bars_h = ax.barh(sorted_results['Algorithm'], sorted_results['Accuracy (%)'],
                 color=colors_h, edgecolor='white', linewidth=1)
for bar, val in zip(bars_h, sorted_results['Accuracy (%)']):
    ax.text(bar.get_width() - 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}%', ha='right', va='center', color='white',
            fontweight='bold', fontsize=10)
ax.set_xlim(min(sorted_results['Accuracy (%)'])-2, 101)
ax.set_title('Fig. 8 — Ranked Model Accuracy (Ascending)', fontsize=12, fontweight='bold')
ax.set_xlabel('Accuracy (%)', fontsize=11)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('fig8_ranked_accuracy.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 9: Random Forest Confusion Matrix ─────────────────────
cm = confusion_matrix(y_test, rf_pred)
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            linewidths=0.3, linecolor='grey', ax=ax, annot_kws={'size': 7})
ax.set_title('Fig. 9 — Random Forest Confusion Matrix (Test Set)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0, labelsize=8)
plt.tight_layout()
plt.savefig('fig9_rf_confusion_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 10: Feature Importance (Random Forest) ────────────────
importances = rf.feature_importances_
feat_imp = pd.Series(importances, index=features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors_fi = sns.color_palette('RdYlGn', len(feat_imp))
bars_fi = ax.barh(feat_imp.index, feat_imp.values, color=colors_fi, edgecolor='white')
for bar, val in zip(bars_fi, feat_imp.values):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
ax.set_title('Fig. 10 — Feature Importance (Random Forest)', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance Score', fontsize=11)
ax.set_ylabel('Feature', fontsize=11)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('fig10_feature_importance.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 11: Classification Report Heatmap (Random Forest) ─────
from sklearn.metrics import precision_recall_fscore_support
prec, rec, f1, sup = precision_recall_fscore_support(y_test, rf_pred)
report_df = pd.DataFrame({
    'Precision': prec, 'Recall': rec, 'F1-Score': f1
}, index=le.classes_)

fig, ax = plt.subplots(figsize=(8, 10))
sns.heatmap(report_df, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, vmin=0.8, vmax=1.0,
            annot_kws={'size': 9})
ax.set_title('Fig. 11 — Random Forest: Precision, Recall & F1-Score per Crop',
             fontsize=12, fontweight='bold', pad=12)
ax.tick_params(axis='x', labelsize=10)
ax.tick_params(axis='y', rotation=0, labelsize=9)
plt.tight_layout()
plt.savefig('fig11_rf_classification_report.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 12: Cross-Validation Scores ──────────────────────────
cv_models = [
    ('LR',  LogisticRegression(max_iter=1000, random_state=42)),
    ('NB',  GaussianNB()),
    ('KNN', KNeighborsClassifier(n_neighbors=3)),
    ('DT',  DecisionTreeClassifier(max_depth=20, random_state=42)),
    ('RF',  RandomForestClassifier(n_estimators=100, random_state=42)),
]
cv_means, cv_stds, cv_labels = [], [], []
for name, model in cv_models:
    scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())
    cv_labels.append(name)
    print(f'{name}: {scores.mean()*100:.2f}% (+/- {scores.std()*100:.2f}%)')

fig, ax = plt.subplots(figsize=(10, 5))
x_pos = range(len(cv_labels))
bar_colors = ['#4472C4', '#ED7D31', '#A9D18E', '#FF0000', '#5B9BD5']
ax.bar(x_pos, [m*100 for m in cv_means], yerr=[s*100 for s in cv_stds],
       color=bar_colors, capsize=6, edgecolor='white', width=0.6)
ax.set_xticks(x_pos)
ax.set_xticklabels(cv_labels, fontsize=11)
ax.set_ylim(min([m*100 for m in cv_means])-3, 101)
ax.set_title('Fig. 12 — 5-Fold Cross-Validation Accuracy (Mean ± Std)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig12_cross_validation.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 13: Violin Plots — Feature vs Top 5 Crops ────────────
top_crops = df['label'].value_counts().head(5).index.tolist()
df_top = df[df['label'].isin(top_crops)]

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, feat in zip(axes, ['N', 'P', 'K']):
    sns.violinplot(x='label', y=feat, data=df_top, ax=ax,
                   palette='Set3', inner='quartile')
    ax.set_title(f'{feat} by Crop', fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)
fig.suptitle('Fig. 13 — Violin Plots: N, P, K Distribution for Top 5 Crops',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig13_violin_plots.png', bbox_inches='tight')
plt.show()

## Step 9 — Crop Prediction (Inference)

In [ ]:
# Full classification report for Random Forest
print('=== Random Forest — Classification Report ===')
print(classification_report(y_test, rf_pred, target_names=le.classes_))

In [ ]:
def predict_crop(N, P, K, temperature, humidity, ph, rainfall):
    """Predict the best crop given soil and climate parameters."""
    sample = scaler.transform([[N, P, K, temperature, humidity, ph, rainfall]])

    rf_crop = le.inverse_transform(rf.predict(sample))[0]
    nb_crop = le.inverse_transform(nb.predict(sample))[0]
    xgb_crop = le.inverse_transform(xgb.predict(sample))[0]

    print('╔══════════════════════════════════════════╗')
    print('║       CROP PREDICTION RESULTS            ║')
    print('╠══════════════════════════════════════════╣')
    print(f'║  Input: N={N}, P={P}, K={K}')
    print(f'║  Temp={temperature}°C, Humidity={humidity}%')
    print(f'║  pH={ph}, Rainfall={rainfall}mm')
    print('╠══════════════════════════════════════════╣')
    print(f'║  Random Forest  : {rf_crop}')
    print(f'║  Naïve Bayes    : {nb_crop}')
    print(f'║  XGBoost        : {xgb_crop}')
    print('╚══════════════════════════════════════════╝')

# Example prediction
predict_crop(N=90, P=42, K=43, temperature=28, humidity=75, ph=6.5, rainfall=120)

In [ ]:
# ── Fig 14: RF Prediction Probability Distribution ────────────
sample_input = scaler.transform([[90, 42, 43, 28, 75, 6.5, 120]])
proba = rf.predict_proba(sample_input)[0]
top_n = 8
top_idx = np.argsort(proba)[::-1][:top_n]

fig, ax = plt.subplots(figsize=(10, 5))
bars_prob = ax.bar(le.classes_[top_idx], proba[top_idx]*100,
                   color=['#FFD700' if i == 0 else '#4472C4' for i in range(top_n)],
                   edgecolor='white')
for bar, val in zip(bars_prob, proba[top_idx]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_title('Fig. 14 — Random Forest: Top 8 Crop Prediction Probabilities\n(Input: N=90, P=42, K=43, Temp=28°C, Humidity=75%, pH=6.5, Rainfall=120mm)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Crop', fontsize=11)
ax.set_ylabel('Probability (%)', fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig14_prediction_probability.png', bbox_inches='tight')
plt.show()

## Summary
| Algorithm | Accuracy |
|---|---|
| Logistic Regression | ~96.36% |
| Naïve Bayes | ~99.55% |
| KNN | ~96.82% |
| Decision Tree | ~98.41% |
| Random Forest | ~99.32% |
| XGBoost | ~98.64% |
| ANN (Keras) | ~99%+ |

**Best Model: Naïve Bayes (99.55%)** — Random Forest is recommended for deployment due to robustness and feature importance support.